# 02 - Feature Engineering and Feature Selection

This notebook builds advanced features and selects the production feature set.


In [ ]:
# Cell 0: Setup and load dataset
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

from ml.pipelines.churn_train import split_by_snapshot

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

DATASET_PATH = ROOT / 'ml' / 'data' / 'churn_training_dataset.csv'
assert DATASET_PATH.exists(), 'Run 01_advanced_eda.ipynb first.'

df = pd.read_csv(DATASET_PATH)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], utc=True)
df['will_order_next_30d'] = df['will_order_next_30d'].astype(int)
print('Loaded:', df.shape)


In [ ]:
# Cell 1: Base numeric feature set
non_features = {'user_id', 'snapshot_date', 'will_order_next_30d'}
base_features = [
    c for c in df.columns
    if c not in non_features and pd.api.types.is_numeric_dtype(df[c])
]

print('Base feature count:', len(base_features))
print(base_features)


In [ ]:
# Cell 2: Advanced feature engineering
feat = df.copy()

# Recency / frequency interactions
feat['orders_30d_per_age'] = feat['orders_30d'] / (feat['customer_age_days'] + 1.0)
feat['revenue_per_order_lifetime'] = feat['revenue_lifetime'] / (feat['order_count_lifetime'] + 1.0)
feat['recency_gap_ratio'] = feat['days_since_last_order'] / (feat['avg_gap_days_lookback'] + 1.0)
feat['promo_x_dinner'] = feat['promo_order_ratio_lookback'] * feat['dinner_order_ratio_lookback']
feat['cancel_x_weekend'] = feat['cancel_ratio_lookback'] * feat['weekend_order_ratio_lookback']

# Stabilized transforms
for col in ['revenue_lookback', 'revenue_lifetime', 'days_since_last_order', 'avg_gap_days_lookback']:
    if col in feat.columns:
        feat[f'log1p_{col}'] = np.log1p(np.clip(feat[col], a_min=0, a_max=None))

# Behavioral flags
feat['is_promo_heavy'] = (feat['promo_order_ratio_lookback'] >= 0.40).astype(int)
feat['is_dinner_heavy'] = (feat['dinner_order_ratio_lookback'] >= 0.50).astype(int)
feat['is_gap_anomaly_like'] = (feat['recency_gap_ratio'] >= 2.0).astype(int)

engineered_path = ROOT / 'ml' / 'data' / 'churn_training_features.csv'
feat.to_csv(engineered_path, index=False)
print('Saved engineered dataset:', engineered_path)
print('Engineered shape:', feat.shape)


In [ ]:
# Cell 3: Time-based split
feature_cols = [
    c for c in feat.columns
    if c not in non_features and pd.api.types.is_numeric_dtype(feat[c])
]

train_df, val_df, test_df = split_by_snapshot(feat)

X_train = train_df[feature_cols]
y_train = train_df['will_order_next_30d']
X_val = val_df[feature_cols]
y_val = val_df['will_order_next_30d']

print('train:', train_df.shape)
print('val:', val_df.shape)
print('test:', test_df.shape)


In [ ]:
# Cell 4: Mutual information ranking
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)

mi_scores = mutual_info_classif(X_train_imp, y_train, random_state=42)
mi_rank = pd.DataFrame({'feature': feature_cols, 'mi': mi_scores}).sort_values('mi', ascending=False)
display(mi_rank.head(25))


In [ ]:
# Cell 5: Permutation importance on candidate set
candidate_features = mi_rank.head(min(30, len(mi_rank)))['feature'].tolist()
idx = [feature_cols.index(c) for c in candidate_features]

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=8,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)
rf.fit(X_train_imp[:, idx], y_train)

perm = permutation_importance(
    rf,
    X_val_imp[:, idx],
    y_val,
    n_repeats=8,
    random_state=42,
    scoring='average_precision',
    n_jobs=-1,
)
perm_rank = pd.DataFrame(
    {'feature': candidate_features, 'perm_importance': perm.importances_mean}
).sort_values('perm_importance', ascending=False)

display(perm_rank.head(25))


In [ ]:
# Cell 6: Final feature set
selected = perm_rank[perm_rank['perm_importance'] > 0]['feature'].tolist()
if len(selected) < 8:
    selected = perm_rank.head(8)['feature'].tolist()
selected = selected[:18]

print('Selected feature count:', len(selected))
print(selected)


In [ ]:
# Cell 7: Save selected feature contract
out = ROOT / 'ml' / 'models' / 'selected_features_notebook.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({'selected_features': selected}, indent=2), encoding='utf-8')
print('Saved selected features:', out)


In [ ]:
# Cell 8: Sanity view of selected features vs target
table = feat[selected + ['will_order_next_30d']].corr(numeric_only=True)['will_order_next_30d']
table = table.drop('will_order_next_30d').sort_values(key=lambda s: s.abs(), ascending=False)
display(table.to_frame('corr_with_target'))


## Next Notebook

Proceed to `03_training_and_evaluation.ipynb`.
